# 第14章 トークナイザーを自作する：BPE

よく出るバイトの組を、くり返し合体させるだけ

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備：文章をバイトにする


In [ ]:
import re, urllib.request

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
train_text = text[:30000]               # 自作版は遅いので、最初の3万字で学習
ids = list(train_text.encode("utf-8"))
print("文字数：", len(train_text), " バイト数：", len(ids))
print("最初の12バイト：", ids[:12])

## ステップ1：ペアを数える


In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(ids)
top = sorted(stats.items(), key=lambda kv: -kv[1])[:5]
for pair, count in top:
    print(pair, count, "回 →", repr(bytes(pair).decode("utf-8", errors="replace")))

## ステップ2：合体させる


In [ ]:
def merge(ids, pair, new_id):
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)     # ペアを見つけたら、新しいIDに置き換えて2つ進む
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

print(merge([5, 6, 6, 7, 9, 6, 7], (6, 7), 99))

## ステップ3：くり返して学習する


In [ ]:
num_merges = 300
merges = {}                     # (a, b) → 新しいID（合体ルール）
vocab = {i: bytes([i]) for i in range(256)}   # ID → バイト列
work = list(ids)
for k in range(num_merges):
    stats = get_stats(work)
    pair = max(stats, key=stats.get)            # いちばん多いペア
    new_id = 256 + k
    work = merge(work, pair, new_id)
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]

print("学習前：", len(ids), "トークン → 学習後：", len(work), "トークン")
print("圧縮率：", round(len(ids) / len(work), 2), "倍")

In [ ]:
learned = [vocab[256 + k].decode("utf-8", errors="replace") for k in range(num_merges)]
print("最初の30個：", learned[:30])
print("最後の30個：", learned[-30:])

## エンコードとデコード


In [ ]:
def decode(token_ids):
    return b"".join(vocab[t] for t in token_ids).decode("utf-8", errors="replace")

def encode(s):
    ids = list(s.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # 学習で「先に」覚えたルール（番号が小さいもの）から順に使う
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                       # もう使えるルールがない
        ids = merge(ids, pair, merges[pair])
    return ids

sample = "先生は奥さんと話をしていた。"
tokens = encode(sample)
print(len(sample.encode("utf-8")), "バイト →", len(tokens), "トークン")
print([decode([t]) for t in tokens])
print(decode(tokens) == sample)

## 本格版：Hugging Faceの tokenizers


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

bpe = Tokenizer(models.BPE())
bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=4000, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
bpe.train_from_iterator(text.split("\n"), trainer=trainer)

enc = bpe.encode("先生は奥さんと話をしていた。")
print(enc.ids)
print([bpe.decode([i]) for i in enc.ids])

## BPEで自作GPTを学習させる


In [ ]:
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/minigpt.py
import math
import torch
from minigpt import GPT

trainer = trainers.BpeTrainer(vocab_size=2000, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
bpe.train_from_iterator(text.split("\n"), trainer=trainer)

n = int(len(text) * 0.9)
train_ids = torch.tensor(bpe.encode(text[:n]).ids)
val_ids = torch.tensor(bpe.encode(text[n:]).ids)
print("検証データ：", len(text[n:]), "文字 →", len(val_ids), "トークン")
print("1トークンあたり平均", round(len(text[n:]) / len(val_ids), 2), "文字")

In [ ]:
def get_batch(data, batch_size, T, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)
    return (torch.stack([data[i:i + T] for i in ix]),
            torch.stack([data[i + 1:i + T + 1] for i in ix]))

@torch.no_grad()
def val_loss_per_token(model, data, T):
    model.eval()
    starts = range(0, len(data) - T - 1, T)
    xs = torch.stack([data[i:i + T] for i in starts])
    ys = torch.stack([data[i + 1:i + T + 1] for i in starts])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = GPT(bpe.get_vocab_size(), block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(1501):
    x, y = get_batch(train_ids, 32, 64, g)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        vl = val_loss_per_token(model, val_ids, 64)
        print(f"ステップ {step:4d}：検証（1トークンあたり）{vl:.3f}")

In [ ]:
per_token = val_loss_per_token(model, val_ids, 64)
per_char = per_token * len(val_ids) / len(text[n:])
print(f"1トークンあたり {per_token:.3f} → 1文字あたり {per_char:.3f}")